# ReparoS Production: OpenNMT-py -> CTranslate2 pipeline

This notebook trains the **ReparoS production** backend on the shared WebSpell production pairs, including Vietnamese Telex/VNI, diacritics, keyboard, boundary, address, and combined errors. It uses OpenNMT-py, not the custom `reparos train` reference trainer. Long-running training starts only in the explicitly marked cell, and checkpoints are written directly to Google Drive.

## 1. Runtime and run configuration

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Hieu31/QU-solution.git'
REPO_ROOT = Path('/content/QU-solution')
DRIVE_ROOT = Path('/content/drive/MyDrive/reparos')
DATA_ZIP = Path('/content/drive/MyDrive/reparos-production-v1.zip')
LOCAL_DATA = REPO_ROOT / 'data/reparos/reparos-production-v1'
RUN_ID = 'opennmt-production-v1'  # Change this for every independent run.
RUN_FULL_TRAIN = False  # Set True only after SMOKE GATE PASSED.
QUALITY_GATE_PASSED = False  # Set automatically by the small learning experiment.
RUN_ROOT = DRIVE_ROOT / RUN_ID
TOKENIZER_ROOT = DRIVE_ROOT / 'tokenizer-production-v1'
CT2_ROOT = DRIVE_ROOT / f'{RUN_ID}-ctranslate2-float32'

print('Python:', sys.version)
subprocess.run(['nvidia-smi'], check=True)
print('Run output:', RUN_ROOT)

## 2. Mount Drive and install the exact backend

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

os.chdir(REPO_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'sync', '--python', '3.11', '--extra', 'reparos-opennmt'], check=True)
VENV_PY = REPO_ROOT / '.venv/bin/python'
REPAROS = REPO_ROOT / '.venv/bin/reparos'
assert VENV_PY.is_file() and REPAROS.is_file()
print('Repository commit:')
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
probe = """
import torch, onmt, ctranslate2, sentencepiece, numpy
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('OpenNMT-py:', getattr(onmt, '__version__', 'unknown'))
print('CTranslate2:', ctranslate2.__version__)
print('NumPy:', numpy.__version__)
assert torch.cuda.is_available(), 'The Python 3.11 training environment cannot see the Colab GPU.'
"""
subprocess.run([str(VENV_PY), '-c', probe], check=True)

## 3. Restore prepared data to local Colab disk

In [ ]:
assert DATA_ZIP.is_file(), f'Missing Drive dataset: {DATA_ZIP}'
(REPO_ROOT / 'data/reparos').mkdir(parents=True, exist_ok=True)
subprocess.run(['unzip', '-q', '-o', str(DATA_ZIP), '-d', str(REPO_ROOT / 'data/reparos')], check=True)

required = [
    LOCAL_DATA / 'base/train.src', LOCAL_DATA / 'base/train.tgt',
    LOCAL_DATA / 'base/validation.src', LOCAL_DATA / 'base/validation.tgt',
]
for path in required:
    assert path.is_file(), f'Missing extracted file: {path}'

for split in ('train', 'validation'):
    src_count = sum(1 for _ in open(LOCAL_DATA / f'base/{split}.src', encoding='utf-8'))
    tgt_count = sum(1 for _ in open(LOCAL_DATA / f'base/{split}.tgt', encoding='utf-8'))
    print(split, 'src=', src_count, 'tgt=', tgt_count)
    assert src_count == tgt_count and src_count > 0

## 4. Train or reuse the 8K SentencePiece tokenizer

In [ ]:
TOKENIZER_MODEL = TOKENIZER_ROOT / 'tokenizer.model'
if TOKENIZER_MODEL.is_file():
    print('Reusing tokenizer:', TOKENIZER_MODEL)
else:
    TOKENIZER_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        str(REPAROS), 'train-tokenizer',
        '--data', str(LOCAL_DATA),
        '--output', str(TOKENIZER_ROOT),
        '--vocab-size', '8000',
    ], check=True)
assert TOKENIZER_MODEL.is_file()
print('Tokenizer:', TOKENIZER_MODEL)

## 5. Mandatory end-to-end smoke gate (2 training steps)

This uses a small aligned subset but executes the real GPU OpenNMT -> checkpoint -> CTranslate2 -> parity path. Continue to the full run only when this cell finishes successfully.

In [ ]:
import itertools, json, shutil

SMOKE_DATA = Path('/content/reparos-smoke-data')
SMOKE_RUN = Path('/content/reparos-smoke-run')
SMOKE_CT2 = Path('/content/reparos-smoke-ctranslate2')
for path in (SMOKE_DATA, SMOKE_RUN, SMOKE_CT2):
    if path.exists():
        shutil.rmtree(path)
(SMOKE_DATA / 'base').mkdir(parents=True)

def copy_aligned_prefix(split, limit):
    src_path = LOCAL_DATA / f'base/{split}.src'
    tgt_path = LOCAL_DATA / f'base/{split}.tgt'
    with src_path.open(encoding='utf-8') as src_stream, tgt_path.open(encoding='utf-8') as tgt_stream:
        src = list(itertools.islice(src_stream, limit))
        tgt = list(itertools.islice(tgt_stream, limit))
    assert len(src) == len(tgt) == limit
    (SMOKE_DATA / f'base/{split}.src').write_text(''.join(src), encoding='utf-8')
    (SMOKE_DATA / f'base/{split}.tgt').write_text(''.join(tgt), encoding='utf-8')

copy_aligned_prefix('train', 2048)
copy_aligned_prefix('validation', 512)

subprocess.run([
    str(REPAROS), 'build-opennmt-config',
    '--data', str(SMOKE_DATA), '--tokenizer', str(TOKENIZER_MODEL),
    '--output', str(SMOKE_RUN), '--train-steps', '2',
    '--valid-steps', '1', '--save-checkpoint-steps', '1',
    '--batch-size', '128', '--bucket-size', '256',
    '--num-workers', '0', '--gpu-rank', '0',
], check=True)
subprocess.run([
    str(REPAROS), 'train-opennmt', '--config', str(SMOKE_RUN / 'opennmt-base.json')
], check=True)

smoke_checkpoints = sorted(SMOKE_RUN.glob('reparos_base_step_*.pt'))
assert smoke_checkpoints, 'Smoke training did not produce a checkpoint.'
SMOKE_CHECKPOINT = smoke_checkpoints[-1]
subprocess.run([
    str(REPAROS), 'export-ctranslate2', '--model', str(SMOKE_CHECKPOINT),
    '--tokenizer', str(TOKENIZER_MODEL), '--output', str(SMOKE_CT2),
    '--compute-type', 'float32', '--trust-checkpoint',
], check=True)

smoke_queries = SMOKE_RUN / 'queries.txt'
query_lines = (SMOKE_DATA / 'base/validation.src').read_text(encoding='utf-8').splitlines()[:4]
smoke_queries.write_text('\n'.join(query_lines) + '\n', encoding='utf-8')
smoke_report = SMOKE_RUN / 'parity.json'
subprocess.run([
    str(REPAROS), 'check-ctranslate2-parity',
    '--checkpoint', str(SMOKE_CHECKPOINT), '--model', str(SMOKE_CT2),
    '--tokenizer', str(TOKENIZER_MODEL), '--queries', str(smoke_queries),
    '--beam-size', '1', '--n-best', '1', '--output', str(smoke_report),
], check=True)
report = json.loads(smoke_report.read_text(encoding='utf-8'))
assert (SMOKE_CT2 / 'model.bin').is_file() and report['queries'] == len(query_lines)
print('SMOKE GATE PASSED:', json.dumps({k: v for k, v in report.items() if k != 'records'}, indent=2))

## 5b. Small learning-quality gate (1,000 steps)

The 2-step smoke above checks plumbing only. This second gate samples balanced clean/noisy pairs across the full Base train split, trains a small run, evaluates a held-out validation sample, and prints real `input | expected | predicted` cases. It does not use the test split.

In [ ]:
import random

QUALITY_DATA = Path('/content/reparos-quality-data')
QUALITY_RUN = Path('/content/reparos-quality-run')
QUALITY_CT2 = Path('/content/reparos-quality-ctranslate2')
for path in (QUALITY_DATA, QUALITY_RUN, QUALITY_CT2):
    if path.exists():
        shutil.rmtree(path)
(QUALITY_DATA / 'base').mkdir(parents=True)

def balanced_reservoir(split, noisy_limit, clean_limit, seed):
    rng = random.Random(seed)
    limits = {True: noisy_limit, False: clean_limit}
    samples = {True: [], False: []}
    seen = {True: 0, False: 0}
    src_path = LOCAL_DATA / f'base/{split}.src'
    tgt_path = LOCAL_DATA / f'base/{split}.tgt'
    with src_path.open(encoding='utf-8') as src_stream, tgt_path.open(encoding='utf-8') as tgt_stream:
        for source, target in zip(src_stream, tgt_stream):
            pair = (source.rstrip('\n'), target.rstrip('\n'))
            bucket = pair[0] != pair[1]
            seen[bucket] += 1
            if len(samples[bucket]) < limits[bucket]:
                samples[bucket].append(pair)
            else:
                position = rng.randrange(seen[bucket])
                if position < limits[bucket]:
                    samples[bucket][position] = pair
    assert all(len(samples[key]) == limits[key] for key in limits), (split, seen)
    result = samples[True] + samples[False]
    rng.shuffle(result)
    return result

def write_pairs(split, pairs):
    (QUALITY_DATA / f'base/{split}.src').write_text(
        ''.join(source + '\n' for source, _ in pairs), encoding='utf-8')
    (QUALITY_DATA / f'base/{split}.tgt').write_text(
        ''.join(target + '\n' for _, target in pairs), encoding='utf-8')

quality_train = balanced_reservoir('train', 4096, 4096, 2026)
quality_validation = balanced_reservoir('validation', 512, 512, 2027)
write_pairs('train', quality_train)
write_pairs('validation', quality_validation)

subprocess.run([
    str(REPAROS), 'build-opennmt-config',
    '--data', str(QUALITY_DATA), '--tokenizer', str(TOKENIZER_MODEL),
    '--output', str(QUALITY_RUN), '--train-steps', '1000',
    '--valid-steps', '250', '--save-checkpoint-steps', '250',
    '--batch-size', '1024', '--bucket-size', '2048',
    '--num-workers', '0', '--warmup-steps', '200', '--gpu-rank', '0',
], check=True)
subprocess.run([
    str(REPAROS), 'train-opennmt', '--config', str(QUALITY_RUN / 'opennmt-base.json')
], check=True)

def step_number(path):
    return int(path.stem.rsplit('_step_', 1)[1])

quality_checkpoints = list(QUALITY_RUN.glob('reparos_base_step_*.pt'))
assert quality_checkpoints
QUALITY_CHECKPOINT = max(quality_checkpoints, key=step_number)
subprocess.run([
    str(REPAROS), 'export-ctranslate2', '--model', str(QUALITY_CHECKPOINT),
    '--tokenizer', str(TOKENIZER_MODEL), '--output', str(QUALITY_CT2),
    '--compute-type', 'float32', '--trust-checkpoint',
], check=True)

quality_predict_script = r'''
import json, os
from pathlib import Path
from reparos.architecture import DecodingConfig
from reparos.serving.ctranslate2 import CTranslate2Predictor
data = Path(os.environ['QUALITY_DATA']) / 'base'
output = Path(os.environ['QUALITY_PREDICTIONS'])
predictor = CTranslate2Predictor(os.environ['QUALITY_CT2'], device='cpu', compute_type='float32')
decoding = DecodingConfig(beam_size=1, num_hypotheses=1, max_decoding_length=100)
sources = (data / 'validation.src').read_text(encoding='utf-8').splitlines()
targets = (data / 'validation.tgt').read_text(encoding='utf-8').splitlines()
with output.open('w', encoding='utf-8') as stream:
    for source, target in zip(sources, targets):
        prediction = predictor.predict(source, decoding=decoding)['top1_query']
        stream.write(json.dumps({'source': source, 'target': target, 'prediction': prediction}, ensure_ascii=False) + '\n')
'''
QUALITY_PREDICTIONS = QUALITY_RUN / 'validation-predictions.jsonl'
quality_env = {**os.environ, 'QUALITY_DATA': str(QUALITY_DATA),
               'QUALITY_CT2': str(QUALITY_CT2), 'QUALITY_PREDICTIONS': str(QUALITY_PREDICTIONS)}
subprocess.run([str(VENV_PY), '-c', quality_predict_script], check=True, env=quality_env)

records = [json.loads(line) for line in QUALITY_PREDICTIONS.read_text(encoding='utf-8').splitlines()]
noisy = [row for row in records if row['source'] != row['target']]
clean = [row for row in records if row['source'] == row['target']]
noisy_correct = sum(row['prediction'] == row['target'] for row in noisy)
clean_preserved = sum(row['prediction'] == row['target'] for row in clean)
metrics = {
    'validation_pairs': len(records),
    'noisy_exact_match': noisy_correct / len(noisy),
    'clean_preservation': clean_preserved / len(clean),
    'overall_exact_match': sum(row['prediction'] == row['target'] for row in records) / len(records),
}
print(json.dumps(metrics, indent=2))
print('\nHELD-OUT NOISY CASES (input | expected | predicted)')
shown = [row for row in noisy if row['prediction'] == row['target']][:10]
shown += [row for row in noisy if row['prediction'] != row['target']][:10]
for row in shown:
    mark = 'OK' if row['prediction'] == row['target'] else 'MISS'
    print(f"[{mark}] {row['source']} | {row['target']} | {row['prediction']}")

assert noisy_correct >= 10, f'Model learned fewer than 10/512 held-out noisy cases: {noisy_correct}'
assert metrics['clean_preservation'] >= 0.50, metrics
QUALITY_GATE_PASSED = True
print('QUALITY GATE PASSED')

## 6. Build and inspect the locked OpenNMT-py Base config

Use a new `RUN_ID` instead of overwriting an earlier run.

In [ ]:
RUN_ROOT.mkdir(parents=True, exist_ok=True)
existing = list(RUN_ROOT.glob('reparos_base_step_*.pt'))
assert not existing, f'RUN_ID already contains checkpoints: {existing[:3]}'

subprocess.run([
    str(REPAROS), 'build-opennmt-config',
    '--data', str(LOCAL_DATA),
    '--tokenizer', str(TOKENIZER_MODEL),
    '--output', str(RUN_ROOT),
    '--train-steps', '100000',
    '--valid-steps', '5000',
    '--save-checkpoint-steps', '5000',
    '--batch-size', '4096',
    '--bucket-size', '8192',
    '--num-workers', '2',
    '--transformer-ff', '512',
    '--dropout', '0.1',
    '--warmup-steps', '4000',
    '--gpu-rank', '0',
], check=True)

import json
resolved = json.loads((RUN_ROOT / 'resolved-architecture.json').read_text(encoding='utf-8'))
print(json.dumps(resolved, ensure_ascii=False, indent=2))

## 7. Fail-fast vocabulary validation

In [ ]:
OPENNMT_CONFIG = RUN_ROOT / 'opennmt-base.json'
subprocess.run([str(REPAROS), 'build-opennmt-vocab', '--config', str(OPENNMT_CONFIG)], check=True)
print('Vocabulary validation passed.')

## 8. TRAIN - long-running cell

This is the only production training cell. It is blocked until `RUN_FULL_TRAIN = True`. Checkpoints are saved directly under `RUN_ROOT` on Drive every 5,000 steps. Do not run the old `reparos train --stage base` command.

In [ ]:
assert QUALITY_GATE_PASSED, 'Full training is locked because the small learning-quality gate has not passed.'
assert RUN_FULL_TRAIN, 'After QUALITY GATE PASSED, explicitly set RUN_FULL_TRAIN = True.'
subprocess.run([str(REPAROS), 'train-opennmt', '--config', str(OPENNMT_CONFIG)], check=True)

## 9. Select and inspect the trained checkpoint

In [ ]:
def checkpoint_step(path: Path) -> int:
    return int(path.stem.rsplit('_step_', 1)[1])

checkpoints = sorted(RUN_ROOT.glob('reparos_base_step_*.pt'), key=checkpoint_step)
assert checkpoints, 'No OpenNMT checkpoint was produced.'
for path in checkpoints:
    print(path.name, f'{path.stat().st_size / 1024**2:.1f} MiB')

# Replace this after comparing validation results across checkpoints.
CHECKPOINT = checkpoints[-1]
print('Selected checkpoint:', CHECKPOINT)

## 10. Convert the trusted OpenNMT checkpoint to CTranslate2 float32

In [ ]:
subprocess.run([
    str(REPAROS), 'export-ctranslate2',
    '--model', str(CHECKPOINT),
    '--tokenizer', str(TOKENIZER_MODEL),
    '--output', str(CT2_ROOT),
    '--compute-type', 'float32',
    '--trust-checkpoint',
], check=True)
print('Converted model:', CT2_ROOT)

## 11. Reference vs CTranslate2 parity

In [ ]:
PARITY_QUERIES = RUN_ROOT / 'parity-queries.txt'
PARITY_QUERIES.write_text('\n'.join([
    'san bay noi bai',
    'ho guom',
    'benh vien bach mai',
    'pho hue hai ba trung',
    'cau giay ha noi',
    'cho ben thanh quan 1',
    'nga tu so',
]) + '\n', encoding='utf-8')

PARITY_REPORT = CT2_ROOT / 'parity.json'
subprocess.run([
    str(REPAROS), 'check-ctranslate2-parity',
    '--checkpoint', str(CHECKPOINT),
    '--model', str(CT2_ROOT),
    '--tokenizer', str(TOKENIZER_MODEL),
    '--queries', str(PARITY_QUERIES),
    '--decoding-config', str(RUN_ROOT / 'decoding-config.json'),
    '--output', str(PARITY_REPORT),
], check=True)
print(PARITY_REPORT.read_text(encoding='utf-8'))

## 12. User-facing predictions from CTranslate2

In [ ]:
prediction_script = r'''
import json, os
from pathlib import Path
from reparos.architecture import DecodingConfig
from reparos.serving.ctranslate2 import CTranslate2Predictor

run_root = Path(os.environ['REPAROS_RUN_ROOT'])
model_root = Path(os.environ['REPAROS_CT2_ROOT'])
queries_path = Path(os.environ['REPAROS_QUERIES'])
decoding = DecodingConfig(**json.loads((run_root / 'decoding-config.json').read_text(encoding='utf-8')))
predictor = CTranslate2Predictor(model_root, device='cpu', compute_type='float32')
for query in queries_path.read_text(encoding='utf-8').splitlines():
    result = predictor.predict(query, decoding=decoding)
    print(f'{query:<30} -> {result["top1_query"]}')
'''
prediction_env = {
    **os.environ,
    'REPAROS_RUN_ROOT': str(RUN_ROOT),
    'REPAROS_CT2_ROOT': str(CT2_ROOT),
    'REPAROS_QUERIES': str(PARITY_QUERIES),
}
subprocess.run([str(VENV_PY), '-c', prediction_script], check=True, env=prediction_env)

## 13. Optional: generate ablation configs only (does not train)

In [ ]:
ABLATION_ROOT = DRIVE_ROOT / 'base-ablation-v1'
subprocess.run([
    str(REPAROS), 'plan-opennmt-ablation',
    '--data', str(LOCAL_DATA),
    '--tokenizer', str(TOKENIZER_MODEL),
    '--output', str(ABLATION_ROOT),
    '--num-workers', '2',
    '--gpu-rank', '0',
], check=True)
print((ABLATION_ROOT / 'ablation-plan.json').read_text(encoding='utf-8'))